## LangChain: Models, Prompts, and Parsers

In [2]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

In [5]:
# account for deprecation of LLM model
import datetime
# Get the current date
current_date = datetime.datetime.now().date()

# Define the date after which the model should be set to "gpt-3.5-turbo"
target_date = datetime.date(2024, 6, 12)

# Set the model variable based on the current date
if current_date > target_date:
    llm_model = "gpt-4"
    
else:
    llm_model = "gpt-3.5-turbo"

print(llm_model)

gpt-4


## ChatAPI: OpenAI

In [10]:
from openai import OpenAI

client = OpenAI()

def get_completion(prompt, model=llm_model):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0, 
    )
    return response.choices[0].message.content


In [11]:
get_completion(prompt = "What is 1+1?")

'2'

In [12]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

style = """American English \
in a calm and respectful tone
"""

prompt = f"""Translate the text \
that is delimited by triple backticks 
into a style that is {style}.
text: ```{customer_email}```
"""

print(prompt)

Translate the text that is delimited by triple backticks 
into a style that is American English in a calm and respectful tone
.
text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse,the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [13]:
response = get_completion(prompt=prompt)

In [14]:
response

"```\nI'm quite upset that my blender lid came off and splattered my kitchen walls with smoothie! To add to the frustration, the warranty doesn't cover the cost of cleaning up my kitchen. I need your help immediately, my friend!\n```"

## ChatAPI: LangChain

In [16]:
from langchain_openai import ChatOpenAI

### Model

In [18]:
# To control the randomness and creativity of the generated
# text by an LLM, use temperature = 0.0
chat = ChatOpenAI(temperature=0.0, model=llm_model)
chat

ChatOpenAI(profile={'name': 'GPT-4', 'release_date': '2023-11-06', 'last_updated': '2024-04-09', 'open_weights': False, 'max_input_tokens': 8192, 'max_output_tokens': 8192, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': False, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x16adb6f50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x16adb7510>, root_client=<openai.OpenAI object at 0x16adb6ad0>, root_async_client=<openai.AsyncOpenAI object at 0x16adb7090>, model_name='gpt-4', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), strea

### Prompt template

In [19]:
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [20]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(template_string)


In [21]:
prompt_template.messages[0].prompt

PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```\n')

In [22]:
prompt_template.messages[0].prompt.input_variables

['style', 'text']

In [23]:
customer_style = """American English \
in a calm and respectful tone
"""

In [24]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [25]:
customer_messages = prompt_template.format_messages(
                    style=customer_style,
                    text=customer_email)

In [26]:
print(type(customer_messages))
print(type(customer_messages[0]))

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>


In [27]:
print(customer_messages[0])

content="Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone\n. text: ```\nArrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n" additional_kwargs={} response_metadata={}


In [31]:
# Call the LLM to translate to the style of the customer message
customer_response = chat.invoke(customer_messages)
print(customer_response.content)

I'm quite upset that my blender lid came off and splattered my kitchen walls with smoothie. To add to the frustration, the warranty doesn't cover the cost of cleaning up my kitchen. I need your help immediately, please.


In [32]:
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""

In [33]:
service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply)

print(service_messages[0].content)

Translate the text that is delimited by triple backticks into a style that is a polite tone that speaks in English Pirate. text: ```Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!
```



In [34]:
service_response = chat.invoke(service_messages)
print(service_response.content)

```Ahoy there, matey, the warranty be not coverin' the cost of swabbin' yer galley, for 'tis ye who be at fault for mishandlin' yer mixin' contraption by forgettin' to secure the top 'fore settin' it to whirl. Hard cheese, me hearty! Until we cross paths again!```
